# **Create Meeting Minutes from an Audio File**

## ENVIRONMENT SETUP & DEPENDENCY INSTALLATION

- GPU runtime recommended (Google Colab)

- Data source: Denver City Council meeting excerpt, could be found here:
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

## IMPORTS & DEPENDENCIES

In [ ]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

## CONFIGURATION & AUTHENTICATION

In [ ]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
AUDIO_MODEL = "gpt-4o-mini-transcribe"

drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

## AUDIO TRANSCRIPTION: HUGGINGFACE WHISPER (OPEN SOURCE)

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
open_source_transcription = result["text"]
print(open_source_transcription)

# ==============================================================================
# 5. AUDIO TRANSCRIPTION: OPENAI WHISPER API
# ==============================================================================
with open(audio_filename, "rb") as audio_file:
    transcription = openai.audio.transcriptions.create(
        model=AUDIO_MODEL,
        file=audio_file,
        response_format="text"
    )
print(transcription)

## TRANSCRIPTION COMPARISON & DISPLAY

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

## PROMPT ENGINEERING FOR MEETING MINUTES

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

## LOCAL LLM QUANTIZATION & TOKENIZER SETUP

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)

## INFERENCE & STREAMED GENERATION (LLAMA 3.2 4-BIT)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config
)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

## OUTPUT DECODING & FINAL DISPLAY

In [ ]:
response = tokenizer.decode(outputs[0])
display(Markdown(response))